In [1]:
import pandas as pd
import numpy as np
import scipy as sp

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mpl_toolkits as mplot3d

import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from scipy.stats.mstats import winsorize


pd.set_option("display.max_columns", None)
pd.reset_option("display.max_rows")
pd.set_option('future.no_silent_downcasting', True)

from utils.categorical_qcut import categorical_qcut

from tqdm import tqdm
tqdm.pandas()

# Data Scoring

In [2]:
scenarios_answers_dict = {
    'M25-1': 'Call for help right away and arrange urgent medical care.',
    'M25-2': 'Use allergy medication if available and call for emergency help.',
    'M25-3': 'Call for the emergency',
    'M25-4': 'Clean wound, stop bleeding, cover it, and seek medical help.',
    'M25-5': 'Stop, sit upright, slow his breathing, and ask for help.',
    'M25-6': 'Block his card and contact the bank helpline right away.',
    'M25-7': 'Hang up and verify with the real police station.',
    'M25-8': 'Ignore the link and report the issue to the official bank helpline/credit bureau.',
    'M25-9': 'Research the scheme independently using RBI or SEBI advisories.',
    'M25-10': 'Check his payment/card records separately and contact his provider if needed.',
    'M25-11': 'Ignore the link, verify status in-app, and enable security features.',
    'M25-12': 'Block UPI and logout from accounts immediately.',
    'M25-13': 'Refuse, disconnect, and verify only via official bank contacts.',
    'M25-14': 'Change password, enable two-factor authentication, and review security.',
    'M25-15': 'Save evidence, secure all accounts, seek support, and prepare a report.',
    'M25-16': 'Collect the messages as evidence and reach out to a counselor or authority for help.',
    'M25-17': 'Discuss things calmly with the friend, or if it continues, talk to a hostel authority.',
    'M25-18': 'Firmly refuse and step away from the situation if necessary, seeking help if required.',
    'M25-19': 'Keep the messages as evidence and reach out to a student helpline or official channel, even if he’s nervous.',
    'M25-20': 'Talk honestly about his feelings with a supportive friend, mentor, or counselor.',
    
    
    'M59-1': 'Take aspirin and call for help or emergency services right away.',
    'M59-2': 'Use allergy medication and get emergency medical help.',
    'M59-3': 'Call emergency services at once and stay put.',
    'M59-4': 'Apply pressure, elevate the hand, and seek urgent help.',
    'M59-5': 'Sit, ask for something sugary, and alert others for support.',

    'M59-6': 'Block his card and freeze the account using the app or helpline.',
    'M59-7': 'Hang up and independently verify via government channels.',
    'M59-8': 'Check his credit report with an official bureau and contact the bank.',
    'M59-9': 'Research independently using official sources/regulators before considering investment.',
    'M59-10': 'Visit the official website or bank branch to check his KYC status.',
    
    'M59-11': "Check his account via the bank’s official app or helpline.",
    'M59-12': 'Change his email password and activate two-factor authentication.',
    'M59-13': 'Refuse the request, hang up, and verify with the bank.',
    'M59-14': 'Regain access through the account recovery process and inform contacts.',
    'M59-15': 'Save evidence, secure all accounts, and seek support to report the breach.',
    
    'M59-16': 'Have a direct but respectful conversation with his supervisor or HR to discuss the situation.',
    'M59-17': 'Save the posts and reach out to a society committee member for help.',
    'M59-18': 'Contact a trusted friend or relative for advice and support.',
    'M59-19': 'Clearly state his limitations and seek shared responsibility from family members.',
    'M59-20': 'Approach his friend in a kind and genuine manner to ask about the situation.',

    
    'M60-1': 'Take aspirin and seek help immediately.',
    'M60-2': 'Use allergy medication and have someone call emergency services.',
    'M60-3': 'Immediately call emergency medical services.',
    'M60-4': 'Apply pressure and elevate the hand, then get help.',
    'M60-5': 'Sit, request something sugary, and alert others.',

    'M60-6': 'Call the official bank helpline to freeze his account and report the issue.',
    'M60-7': 'Hang up and verify directly with the Income Tax Department.',
    'M60-8': 'Contact the concerned bank and credit bureau to report the issue.',
    'M60-9': 'Decline and independently research the investment scheme.',
    'M60-10': 'Confirm with his bank in person or via the official website.',

    'M60-11': 'Check through the official bank app or in person.',
    'M60-12': 'Change his password and set up two-factor authentication.',
    'M60-13': 'Refuse, disconnect, and verify with his bank directly.',
    'M60-14': 'Regain access through account recovery process and alert contacts.',
    'M60-15': 'Collect evidence, secure accounts, and enlist trusted help for reporting.',

    'M60-16': 'Contact a trusted friend or community group to reconnect socially.',
    'M60-17': 'Calmly ask a trusted neighbor or official for their perspective and, if necessary, address the matter privately.',
    'M60-18': 'Open a conversation with a family member he trusts to seek understanding and resolution.',
    'M60-19': "Tell his family honestly that he needs help and identify specific support he’d appreciate.",
    'M60-20': 'Ask a friendly neighbor in a polite, non-confrontational way if he missed an invite.',


    'W25-1': 'Call for help from hostel staff or friends, and get to a campus clinic or similar immediately.',
    'W25-2': 'Use her allergy medication (if available) and seek emergency help immediately.',
    'W25-3': 'Call for the emergency',
    'W25-4': 'Apply pressure, clean the wound, and go to a clinic.',
    'W25-5': 'Stop, sit up, slow her breathing, and call for help.',

    'W25-6': 'Block her card and report to the bank immediately via app/helpline.',
    'W25-7': 'Hang up, then verify the demand with official government contacts.',
    'W25-8': 'Report identity theft to the bank and credit bureau.',
    'W25-9': 'Research the investment through official sources or regulators',
    'W25-10': 'Do not open the PDF; verify independently using her bank app or statement.',

    'W25-11': "Don't click the link; verify from the app and enable security features.",
    'W25-12': 'Block UPI and disable transactions immediately',
    'W25-13': 'Refuse, disconnect, and verify with the official app helpline.',
    'W25-14': 'Immediately change her password and activate security settings.',
    'W25-15': 'Save evidence and alert someone she trusts while preparing to report.',

    'W25-16': "Save screenshots as evidence and approach a counselor or a trusted adult for help.",
    'W25-17': 'Have a calm one-on-one conversation with her roommate or talk to the hostel warden.',
    'W25-18': 'Firmly tell the group she’s not interested and, if needed, leave the situation.',
    'W25-19': 'Save the messages and reach out to student support or university authorities for help.',
    'W25-20': 'Speak with a counselor or someone she trusts about feeling left out and work through her emotions.',




    'W59-1': 'Take an aspirin and call for help/emergency services immediately.',
    'W59-2': 'Use her emergency allergy medication and call for medical help.',
    'W59-3': 'Call emergency medical services at once.',
    'W59-4': 'Apply direct pressure, elevate hand and call for help.',
    'W59-5': 'Sit, ask someone for sugar, and alert for help.',

    'W59-6': 'Block her card and report the transactions to the bank immediately.',
    'W59-7': 'Hang up and independently contact the Income Tax Department.',
    'W59-8': 'Check her credit report through official channels and alert her bank.',
    'W59-9': 'Research the scheme independently using official sources.',
    'W59-10': 'Visit her bank’s official website or branch to check the KYC status.',
    
    'W59-11': 'Ignore the link, check through the official app or website.',
    'W59-12': 'Change her password and turn on two-factor authentication.',
    'W59-13': 'Refuse and verify with official IT support.',
    'W59-14': 'Change her password and secure her account.',
    'W59-15': 'Save evidence, secure accounts, and reach out for support while reporting.',

    'W59-16': 'Reach out directly to her supervisor or HR to discuss her concerns.',
    'W59-17': 'Save the hurtful messages as evidence and request intervention from building management.',
    'W59-18': 'Reach out to a trusted friend or counselor and share her feelings honestly.',
    'W59-19': 'Talk openly and respectfully with her in-laws about boundaries and shared responsibilities.',
    'W59-20': 'Privately and calmly seek clarification, possibly requesting mediation.',


    'W60-1': 'Chew aspirin and call for help immediately.',
    'W60-2': 'Stay still and shout for assistance.',
    'W60-3': 'Stop reading, call for help, and get to the hospital.',
    'W60-4': 'Sit, stay safe, and call for help immediately.',
    'W60-5': 'Call the pharmacy and verify the pills.',

    'W60-6': 'Immediately call the official bank helpline to report and freeze her account.',
    'W60-7': 'Hang up, then call her bank using the number listed on her bank documents.',
    'W60-8': 'Request a formal Fraud Alert on her credit profile through her bank.',
    'W60-9': 'Refuse politely and research the chit fund through official channels.',
    'W60-10': 'Visit her bank’s official website or branch to clarify.',

    'W60-11': 'Verify with someone she trusts and ignore the suspicious message.',
    'W60-12': 'Ask a trusted family member or neighbor for help reading the instructions.',
    'W60-13': 'Refuse and independently verify with someone she knows.',
    'W60-14': 'Set the phone to airplane mode and seek help to check for malware.',
    'W60-15': 'Change her password and secure the account.',
    
    'W60-16': 'Reach out to someone she trusts and share how she’s feeling.',
    'W60-17': 'Seek clarification from a trusted member or speak privately to address the rumors.',
    'W60-18': 'Reflect for a while and then gently bring up her feelings with a family member.',
    'W60-19': 'Let her family know about her concerns and clearly request specific help where needed.',
    'W60-20': 'Gently ask a trusted neighbor if there was a reason she was left out.',


}

# Data Cleaning

In [4]:
# importing
import_path = f"../../data_cleaned/india/dynata_pilot/"
data = pd.read_csv(f"{import_path}/DynataScenarioPiloting_August 20.csv")

questions_df = data[:1]

data = data[2:].copy()
data["Q2"] = data["Q2"].astype(float)

# scoring
print("Following questions have no correct answers in the demographic:")
for key, value in scenarios_answers_dict.items():
    data[key] = np.where(data[key].str.strip() == value.strip(), 1, 0)
    if data[key].sum() == 0:
        print(f"{key}: sum = {data[key].sum()}, count = {data[key].count()}")

# removing those that were filtered out due to demographic misalignment
data = data.copy()
data["score_sum"] = data.loc[:, list(scenarios_answers_dict.keys())].sum(axis = 1)
data = data.loc[ data["score_sum"]!= 0].copy()

# Removing preview responses
data = data.loc[ data["Status"] != "Survey Preview"].copy()

# <0.5 median time of completion filtered --> already done for this dataset
data["Duration (in seconds)"] = data["Duration (in seconds)"].astype("Int64")
median = data["Duration (in seconds)"].astype("Int64").median()
data = data.loc[ data["Duration (in seconds)"] >= median*0.5].copy()
print("\nPost removing <0.5 median time responses:", len(data["ResponseId"]))

output_col = ["ResponseId", "Q1", "Q2", "Q3", "Q4a"] + list(scenarios_answers_dict.keys())
output = data.loc[:, output_col].copy()
output.to_excel("dynata_pilot_cleaned_200825.xlsx")

Following questions have no correct answers in the demographic:
M60-20: sum = 0, count = 189

Post removing <0.5 median time responses: 112


In [5]:
scenarios_questions_dict = {}
for key in scenarios_answers_dict.keys():
    scenarios_questions_dict[key] = questions_df[key][0]

In [6]:
def sample_prep(data, sex, age):
    sex2 = "Male" if sex == "M" else "Female"
    if age == 25:
        age_upper = 25
        age_lower = 18
    elif age == 59:
        age_upper = 59
        age_lower = 26
    else:
        age_upper = 122
        age_lower = 60
        
    sample = data.dropna(subset = ["Q1", "Q2"]).loc[ (data["Q1"] == sex2) & ((data["Q2"] >= age_lower) & (data["Q2"] <= age_upper))].copy()
    return sample

def question_analysis(data, sex, age):

    sample = sample_prep(data, sex, age)
    if sex not in {"M", "W"}:
        raise ValueError("sex not applicable, input one of 'M' or 'W'")
    if age not in {25, 59, 60}:
        raise ValueError("age not applicable, input one of 25, 59, 60")

    col_prefix = f"{sex}{age}-"
    focal_columns = [x for x in sample.columns if x.startswith(col_prefix)]
    
    for col in focal_columns:
        print(f"{col}:{ sample[col].sum() / sample[col].count() }")


In [7]:
# Demographics
for sex in {"M", "W"}:
    for age in {25, 59, 60}:
        sex2 = "Male" if sex == "M" else "Female"
        if age == 25:
            age2 = "18-25"
        elif age == 59:
            age2 = "26-59"
        else:
            age2 = "60+"

        print(f"{sex2:<7} {age2:<7}: {len(sample_prep(data, sex, age))}")

Female  18-25  : 17
Female  26-59  : 23
Female  60+    : 5
Male    18-25  : 25
Male    26-59  : 29
Male    60+    : 13


# Score Averages for each demoghraphic

In [8]:
for sex in ["M", "W"]:
    for age in [25, 59, 60]:
        sex2 = "Male" if sex == "M" else "Female"
        if age == 25:
            age2 = "18-25"
        elif age == 59:
            age2 = "26-59"
        else:
            age2 = "60+"

        print(f"\n\n{sex2:<7} {age2:<7}: count: {len(sample_prep(data, sex, age))}")
        question_analysis(data, sex, age)



Male    18-25  : count: 25
M25-1:0.8
M25-2:0.76
M25-3:0.52
M25-4:0.84
M25-5:0.88
M25-6:0.76
M25-7:0.76
M25-8:0.76
M25-9:0.68
M25-10:0.52
M25-11:0.68
M25-12:0.48
M25-13:0.72
M25-14:0.88
M25-15:0.48
M25-16:0.76
M25-17:0.84
M25-18:0.64
M25-19:0.6
M25-20:0.6


Male    26-59  : count: 29
M59-1:0.9655172413793104
M59-2:1.0
M59-3:0.7586206896551724
M59-4:0.896551724137931
M59-5:0.9310344827586207
M59-6:0.896551724137931
M59-7:0.5862068965517241
M59-8:0.6551724137931034
M59-9:0.8620689655172413
M59-10:1.0
M59-11:0.8620689655172413
M59-12:0.8620689655172413
M59-13:0.896551724137931
M59-14:0.3103448275862069
M59-15:0.7241379310344828
M59-16:0.8275862068965517
M59-17:0.6551724137931034
M59-18:0.2413793103448276
M59-19:0.4482758620689655
M59-20:0.4827586206896552


Male    60+    : count: 13
M60-1:0.6153846153846154
M60-2:0.8461538461538461
M60-3:0.6923076923076923
M60-4:0.5384615384615384
M60-5:0.9230769230769231
M60-6:0.8461538461538461
M60-7:0.9230769230769231
M60-8:0.5384615384615384
M60-9:0